In [2]:
## Run this cell to install the necessary libraries for using HuggingFace Transformers.
"""!pip install transformers
## for accessing various LLMs available in HuggingFace
!pip install accelerate
## for faster inference, especially on GPU
!pip install sentencepiece
## for tokenization that many models require"""


'!pip install transformers\n## for accessing various LLMs available in HuggingFace\n!pip install accelerate\n## for faster inference, especially on GPU\n!pip install sentencepiece\n## for tokenization that many models require'

In [43]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch
import transformers

In [50]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Quick inference using pipeline

In [5]:
pipe = pipeline(
    "text-generation", ## task type
    model  = "openai-community/gpt2", ## the model from which we want to generate text
    device_map = device, ## automatically selects the device (CPU/GPU) for inference
    torch_dtype = "auto" ## automatically selects the data type for tensors (float16 for GPU, float32 for CPU)

)

response = pipe("What is AI?", max_new_tokens=100) ## This returns a dictionary of all its responses but the default value of the number of responses is one. So we extract the first response from the dictionary for our use.
## The pipe fucntion is for giving input to the model
## max_new_tokens specifies the maximum legth of the output response

print(response[0]["generated_text"])

Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What is AI? As I've mentioned previously, there may be AI experiments in our future, but many experiments might not use existing technologies anymore

I have been looking at the best ways to understand AI for some time now, and with a great deal of effort, I ended up adopting the methods in this post and making the code available to anyone with Python at a cost of around $300+ or more.

In this post, I have a list of the best techniques and tools that I have used (


## Manual tokenisation and then prompting

In [6]:
model_id = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map = device
)

prompt = "Write a program that prints fibonacci heap in python."

inputs = tokenizer(prompt, return_tensors = "pt").to(device) ## "pt" is used so that it returns a pytorch tensor and not any list or array.

outputs = model.generate(
    **inputs,
    max_new_tokens = 100,
    do_sample = True, ## do_sample is used to sample the next token from the distribution of the model and not just take the argmax of the distribution which is the most likely token and makes repetition in the output.
    temperature = 0.7 ## temperature is used to control the randomness of the output. Lower temperature means less randomness and higher temperature means more randomness. Should keep it between 0.7 and 1.0 for good results.
)

print(tokenizer.decode(outputs[0], skip_special_tokens = True)) ## special tokens are used by the model for various purposes like padding, start of sentence, end of sentence, etc. We skip them to get the actual output.

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Write a program that prints fibonacci heap in python.

import random import time import random import collections. collections. ArrayList import random import time import sys import time. sleep

After a while the program will ask itself "what is the point of this program?".

There is, however, one more way to do this:

This is where I want the program to stop.

So I need to stop the program and ask the code to stop.

I am going to start by doing something like this:



### For chat specific model like LLama, Mistral, following can be used

In [7]:
"""prompt_template = [
    {
        "role": "system", ## The system message is used to set the context for the conversation.
        "content": "You are a pirate."
    },
    {
        "role": "user", ## The user message is used to provide input to the model.
        "content": "Who is Monkey D. Luffy?"
    }
    ## we can also add like "role": "assistant", "content":"blah blah blah" and then again a user so as to show a conntinued conversation.
]

tokenized = tokenizer.apply_chat_template(
    prompt_template,
    add_generation_prompt = False, ## This is used to add a generation prompt to the input so that the model knows that it has to generate a response.
    tokenize = True, ## This is used to tokenize the input so that it can be fed to the model.
    padding = True,
    return_tensors = "pt"
)

print(tokenized) ## This will just print the tokenized text and not the respnse of model."""

'prompt_template = [\n    {\n        "role": "system", ## The system message is used to set the context for the conversation.\n        "content": "You are a pirate."\n    },\n    {\n        "role": "user", ## The user message is used to provide input to the model.\n        "content": "Who is Monkey D. Luffy?"\n    }\n    ## we can also add like "role": "assistant", "content":"blah blah blah" and then again a user so as to show a conntinued conversation.\n]\n\ntokenized = tokenizer.apply_chat_template(\n    prompt_template,\n    add_generation_prompt = False, ## This is used to add a generation prompt to the input so that the model knows that it has to generate a response.\n    tokenize = True, ## This is used to tokenize the input so that it can be fed to the model.\n    padding = True,\n    return_tensors = "pt"\n)\n\nprint(tokenized) ## This will just print the tokenized text and not the respnse of model.'

### For non-chat models like GPT-2, use the following

In [8]:
prompt_template = "You are a pirate.\
    User: Who is Monkey D. Luffy?\
        Assistant: Captain, I'm not sure.\
            User: Don't you know about One Piece?\
                Assistant:"

inputs = tokenizer(prompt_template, return_tensors = "pt").to(device)
outputs = model.generate(
    **inputs,
    max_new_tokens = 100,
    do_sample = True,
    temperature = 0.7
)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


You are a pirate.    User: Who is Monkey D. Luffy?        Assistant: Captain, I'm not sure.            User: Don't you know about One Piece?                Assistant: I'm sorry, Captain.                  User: I'm not sure.                User: We are pirates.                     User: Are you using a weapon... User: What's that?               


# Finetuning GPT-2

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from datasets import load_dataset

In [10]:
devie = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

#### Loading the model

In [11]:
mode_id = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map = device
)

#### Dataset on which we finetune model

In [12]:
data = load_dataset("dbpedia_14", split="train") ## This loads the DBpedia dataset which is a dataset of Wikipedia articles classified into 14 categories. It is used for text classification tasks.
data

Using the latest cached version of the dataset since dbpedia_14 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'dbpedia_14' at C:\Users\mayan\.cache\huggingface\datasets\dbpedia_14\dbpedia_14\0.0.0\9abd46cf7fc8b4c64290f26993c540b92aa145ac (last modified on Sun Jun  8 03:36:51 2025).


Dataset({
    features: ['label', 'title', 'content'],
    num_rows: 560000
})

In [33]:
data = data.shuffle()
data

Dataset({
    features: ['label', 'title', 'content'],
    num_rows: 560000
})

In [34]:
labels = data.features["label"].names
labels

['Company',
 'EducationalInstitution',
 'Artist',
 'Athlete',
 'OfficeHolder',
 'MeanOfTransportation',
 'Building',
 'NaturalPlace',
 'Village',
 'Animal',
 'Plant',
 'Album',
 'Film',
 'WrittenWork']

In [35]:
data.to_pandas()['label'].value_counts()

label
4     40000
2     40000
10    40000
9     40000
11    40000
13    40000
6     40000
12    40000
7     40000
8     40000
0     40000
1     40000
3     40000
5     40000
Name: count, dtype: int64

#### Seeing the original model performance

In [ ]:
for sample in data:
    content = sample["content"]
    trg_label = labels[sample["label"]]

    prompt = f"You are an AI assistant who has to select that to which category the text belongs. Output only one category out of the given one and nothing else.\n Text: {content}\n Choose any one of the following categories: {', '.join(labels)}\n Category:"
    
    inputs = tokenizer(prompt, return_tensors = "pt").to(device)
    output = model.generate(
        **inputs,
        max_new_tokens = 10,
        do_sample = False
    )
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    print(response)
    print(f"target label: {trg_label}\n\n")
    total += 1
    if total > 5:
        break

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


You are an AI assistant who has to select that to which category the text belongs. Output only one category out of the given one and nothing else.
 Text:  John Hillen (born 3 February 1966) is an American business executive and the former Assistant Secretary of State for Political-Military Affairs nominated by President George W. Bush who served from October 11 2005 until January 11 2007. He served as President & CEO of Sotera Defense Solutions formerly Global Defense Technology & Systems Inc. (GTEC) from 2008 - June 18 2013. While at Sotera he took the company public in November 2009. John currently serves on Sotera's Board of Advisors.
 Choose any one of the following categories: Company, EducationalInstitution, Artist, Athlete, OfficeHolder, MeanOfTransportation, Building, NaturalPlace, Village, Animal, Plant, Album, Film, WrittenWork
 Category: 

The following categories are available for you
target label: OfficeHolder


You are an AI assistant who has to select that to which categ

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


You are an AI assistant who has to select that to which category the text belongs. Output only one category out of the given one and nothing else.
 Text:  Myristica ceylanica is a species of plant in the Myristicaceae family. It is endemic to Sri Lanka.
 Choose any one of the following categories: Company, EducationalInstitution, Artist, Athlete, OfficeHolder, MeanOfTransportation, Building, NaturalPlace, Village, Animal, Plant, Album, Film, WrittenWork
 Category: 

The following categories are available:

target label: Plant




Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


You are an AI assistant who has to select that to which category the text belongs. Output only one category out of the given one and nothing else.
 Text:  Histioea falerina is a moth of the Arctiidae family. It was described by Druce in 1907. It is found in Peru.
 Choose any one of the following categories: Company, EducationalInstitution, Artist, Athlete, OfficeHolder, MeanOfTransportation, Building, NaturalPlace, Village, Animal, Plant, Album, Film, WrittenWork
 Category: 

The following categories are available:

target label: Animal


You are an AI assistant who has to select that to which category the text belongs. Output only one category out of the given one and nothing else.
 Text:  City High is the debut and only album by R&B trio City High. It was released on May 22 2001.
 Choose any one of the following categories: Company, EducationalInstitution, Artist, Athlete, OfficeHolder, MeanOfTransportation, Building, NaturalPlace, Village, Animal, Plant, Album, Film, WrittenWork
 Ca

In [ ]:
from peft import LoraConfig, get_peft_model

def parameters(model):
    trainable_params = 0
    all_params = 0
    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"Trainable params: {trainable_params} out of total params: {all_params} that is {trainable_params / all_params * 100: .2f}% of total.\n Note this is percentage and not fraction.")

def merge_columns(row_sample):
    row_sample["prediction"] = f"You are an AI assistant who has to select that to which category the text belongs. Output only one category out of the given one and nothing else.\n Text: {row_sample["content"]}\n Choose any one of the following categories: {', '.join(labels)}\n Category: {labels[row_sample["label"]]}"
    return row_sample

tokenizer.pad_token = tokenizer.eos_token ## This is used to set the padding token to the end of sentence token so that the model knows that it has to pad the input to the end of sentence token.

lora_config = LoraConfig(
    r = 16, ## This is the rank of the low-rank decomposition. It controls the number of parameters that are updated during training.
    lora_alpha = 32, ## This is the scaling factor for the low-rank decomposition. It controls the strength of the low-rank decomposition.
    lora_dropout = 0.05, ## This is the dropout rate for the low-rank decomposition. It controls the amount of dropout applied to the low-rank decomposition.
    bias = "none", ## This is used to specify whether to use bias in the low-rank decomposition or not. "none" means no bias, "all" means bias for all layers, and "lora_only" means bias only for the low-rank decomposition.
    task_type = "SEQ_CLS", ## This is used to specify the type of task for which the model is being trained."SEQ_CLS" means sequential classification which we need here as our main goal is classification. "CAUSAL_LM" means causal language modeling which is used for next token prediction.
)

model = get_peft_model(model, lora_config) ## This is used to get the model with the low-rank decomposition applied to it.

parameters(model)

lora_data = data.map(merge_columns)
lora_data = lora_data.map(lambda sample: tokenizer(sample["prediction"]), batched = True) ## batched = True means that the function will be applied to the entire batch of data at once instead of one sample at a time. This is more efficient and faster.

trainer = transformers.Trainer(
    model = model,
    train_dataset = lora_data,
    args = transformers.TrainingArguments(
        per_device_train_batch_size = 4, ## This is the batch size for training. It is the number of samples that will be processed at once.
        gradient_accumulation_steps = 4, ## This is the number of steps for which the gradients will be accumulated before updating the model parameters. It is used to increase the effective batch size without increasing the memory usage.
        warmup_steps = 100, ## This is the number of steps for which the learning rate will be increased linearly from 0 to the initial learning rate. It is used to prevent the model from diverging at the beginning of training.
        max_steps = 500, ## This is the maximum number of steps for which the model will be trained. It is used to limit the training time.
        learning_rate = 2e-4, ## This is the initial learning rate for the optimizer. It is used to control the step size for updating the model parameters.
        logging_steps = 10, ## This is the number of steps after which the training progress will be logged. It is used to monitor the training progress.
        output_dir = "outuputs", ## This is the directory where the model checkpoints and training logs will be saved.
        auto_find_batch_size = True ## This is used to automatically find the best batch size for training. It is used to optimize the training process by finding the best batch size that fits in the available memory without causing out of memory errors.
    ),
    data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm = False)
)
model.config.use_cache = False ## This is used to disable the cache for the model. It is used to prevent the model from using the cache during training which can cause issues with the low-rank decomposition.
trainer.train()

c:\Users\mayan\anaconda3\envs\ai\Lib\site-packages\peft\mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
c:\Users\mayan\anaconda3\envs\ai\Lib\site-packages\peft\tuners\tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Trainable params: 589824 out of total params: 125029632 that is  0.47% of total.
 Note this is percentage and not fraction.


Map: 100%|██████████| 560000/560000 [01:51<00:00, 5037.82 examples/s] 


Step,Training Loss
10,4.653900
20,4.612900
30,4.522400
40,4.478200
50,4.369200
60,4.135500
70,3.829300
80,3.421200
90,2.944600
100,2.570500


RuntimeError: 
            Some tensors share memory, this will lead to duplicate memory on disk and potential differences when loading them again: [{'base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.transformer.wte.weight', 'base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.lm_head.weight'}].
            A potential way to correctly save your model is to use `save_model`.
            More information at https://huggingface.co/docs/safetensors/torch_shared_tensors
            

In [49]:
total = 0
for sample in data:
    content = sample["content"]
    trg_label = labels[sample["label"]]

    prompt = f"You are an AI assistant who has to select that to which category the text belongs. Output only one category out of the given one and nothing else.\n Text: {content}\n Choose any one of the following categories: {', '.join(labels)}\n Category:"
    
    inputs = tokenizer(prompt, return_tensors = "pt").to(device)
    output = model.generate(
        **inputs,
        max_new_tokens = 10,
        do_sample = False
    )
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    print(response)
    print(f"target label: {trg_label}\n\n")
    total += 1
    if total > 5:
        break

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


You are an AI assistant who has to select that to which category the text belongs. Output only one category out of the given one and nothing else.
 Text:  John Hillen (born 3 February 1966) is an American business executive and the former Assistant Secretary of State for Political-Military Affairs nominated by President George W. Bush who served from October 11 2005 until January 11 2007. He served as President & CEO of Sotera Defense Solutions formerly Global Defense Technology & Systems Inc. (GTEC) from 2008 - June 18 2013. While at Sotera he took the company public in November 2009. John currently serves on Sotera's Board of Advisors.
 Choose any one of the following categories: Company, EducationalInstitution, Artist, Athlete, OfficeHolder, MeanOfTransportation, Building, NaturalPlace, Village, Animal, Plant, Album, Film, WrittenWork
 Category: OfficeHolder
 Category: MeanOfTransportation
target label: OfficeHolder


You are an AI assistant who has to select that to which category 

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


You are an AI assistant who has to select that to which category the text belongs. Output only one category out of the given one and nothing else.
 Text:  Myristica ceylanica is a species of plant in the Myristicaceae family. It is endemic to Sri Lanka.
 Choose any one of the following categories: Company, EducationalInstitution, Artist, Athlete, OfficeHolder, MeanOfTransportation, Building, NaturalPlace, Village, Animal, Plant, Album, Film, WrittenWork
 Category: Plant, Animal, Album, WrittenWork
 Category
target label: Plant


You are an AI assistant who has to select that to which category the text belongs. Output only one category out of the given one and nothing else.
 Text:  Histioea falerina is a moth of the Arctiidae family. It was described by Druce in 1907. It is found in Peru.
 Choose any one of the following categories: Company, EducationalInstitution, Artist, Athlete, OfficeHolder, MeanOfTransportation, Building, NaturalPlace, Village, Animal, Plant, Album, Film, WrittenWo

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


You are an AI assistant who has to select that to which category the text belongs. Output only one category out of the given one and nothing else.
 Text:  City High is the debut and only album by R&B trio City High. It was released on May 22 2001.
 Choose any one of the following categories: Company, EducationalInstitution, Artist, Athlete, OfficeHolder, MeanOfTransportation, Building, NaturalPlace, Village, Animal, Plant, Album, Film, WrittenWork
 Category: Album, Film, WrittenWork
 Category: Album
target label: Album


You are an AI assistant who has to select that to which category the text belongs. Output only one category out of the given one and nothing else.
 Text:  Cindy Denby is a Republican politician from Michigan currently serving in the Michigan House of Representatives. She also served for 16 years on the Handy Township Board of Trustees: eight years as township clerk and eight as supervisor.Denby is a member of numerous community boards and organizations and is the forme